# Colab-Ready End-to-End AI Research-Paper Summarization

This notebook implements Problem Statement 5 as a runnable end-to-end prototype using an actual research paper:

- PDF text extraction
- Cleaning and section detection
- Long-document chunking
- Abstractive summarization with a T5/BART/PEGASUS model
- Structured extraction of objective, methodology, dataset, findings, and conclusion
- ROUGE-1, ROUGE-2, ROUGE-L, BLEU, and perplexity
- Kaggle ArXiv Scientific Paper Dataset loader with paper/abstract pairs

The notebook automatically downloads the sample ArXiv paper. If the download fails, it prompts for a PDF upload. The Kaggle ArXiv loader is included for optional dataset-based evaluation.

In [ ]:
# Install missing packages into a local notebook directory, not system Python.
import importlib.util
import subprocess
import sys
from pathlib import Path

LOCAL_PACKAGE_DIR = Path('.notebook_packages')
LOCAL_PACKAGE_DIR.mkdir(exist_ok=True)
if str(LOCAL_PACKAGE_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(LOCAL_PACKAGE_DIR.resolve()))

if importlib.util.find_spec('pymupdf') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--target', str(LOCAL_PACKAGE_DIR), 'pymupdf'])

missing_transformer_packages = [
    package for module, package in {'transformers': 'transformers', 'sentencepiece': 'sentencepiece', 'torch': 'torch'}.items()
    if importlib.util.find_spec(module) is None
]
if missing_transformer_packages and sys.version_info >= (3, 14):
    raise RuntimeError(
        'This notebook requires Python 3.11-3.13 for the Transformer model. '
        f'Your kernel is Python {sys.version_info.major}.{sys.version_info.minor}. '
        'Create/select a Python 3.13 Jupyter kernel, then rerun the notebook.'
    )
if missing_transformer_packages:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--target', str(LOCAL_PACKAGE_DIR), *missing_transformer_packages])
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Transformer dependencies could not be installed. Use a Python 3.11-3.13 kernel.') from exc

import math
import re
import urllib.request
from collections import Counter
from typing import Dict, List

import pymupdf

PDF_PATH = Path('attention_is_all_you_need.pdf')
PAPER_URL = 'https://arxiv.org/pdf/1706.03762'
if not PDF_PATH.exists():
    try:
        print('Downloading sample research paper...')
        urllib.request.urlretrieve(PAPER_URL, PDF_PATH)
    except Exception as download_error:
        print('Automatic download unavailable:', download_error)
if not PDF_PATH.exists():
    try:
        from google.colab import files
        print('Upload the research-paper PDF to continue.')
        uploaded = files.upload()
        if uploaded:
            PDF_PATH = Path(next(iter(uploaded)))
    except ImportError:
        pass
MODEL_NAME = 'google/flan-t5-base'  # T5 encoder-decoder model required by the assignment
MAX_CHUNK_WORDS = 650
# Set this to the downloaded Kaggle dataset directory for multi-paper evaluation.
DATASET_DIR = Path('arxiv_dataset')

assert PDF_PATH.exists(), f'Research paper not found: {PDF_PATH}'
print('Input:', PDF_PATH)

def load_kaggle_pairs(dataset_dir: Path, limit: int = 5):
    """Load paper/abstract pairs from the Kaggle ArXiv Scientific Paper Dataset."""
    import csv
    import json
    pairs = []
    def normalize_row(row):
        return {str(key).strip().lower().replace(' ', '_').replace('-', '_').replace('\ufeff', ''): str(value or '').strip()
                for key, value in row.items() if key is not None}
    def first(row, *names):
        for name in names:
            value = row.get(name, '')
            if value: return value
        return ''
    for csv_path in dataset_dir.rglob('*.csv') if dataset_dir.exists() else []:
        with csv_path.open(encoding='utf-8', errors='ignore', newline='') as handle:
            reader = csv.DictReader(handle)
            print(f'Reading {csv_path.name}; columns: {reader.fieldnames}')
            for raw_row in reader:
                row = normalize_row(raw_row)
                paper = first(row, 'article', 'paper', 'text', 'full_text', 'article_text', 'body', 'content')
                abstract = first(row, 'abstract', 'summary', 'target', 'abstract_text', 'summaries')
                # Some Kaggle ArXiv files contain metadata plus abstracts, not full text.
                if not paper and abstract:
                    paper = f"{first(row, 'title', 'paper_title')}. {abstract}"
                if paper and abstract:
                    pairs.append({'paper': paper, 'reference': abstract, 'source': csv_path.name})
                    if len(pairs) >= limit: return pairs
    for json_path in dataset_dir.rglob('*.json') if dataset_dir.exists() else []:
        try:
            records = json.loads(json_path.read_text(encoding='utf-8', errors='ignore'))
        except Exception:
            continue
        if isinstance(records, dict): records = records.get('data', records.get('papers', []))
        for row in records if isinstance(records, list) else []:
            if not isinstance(row, dict): continue
            row = normalize_row(row)
            paper = first(row, 'article', 'paper', 'article_text', 'full_text', 'body', 'content')
            abstract = first(row, 'abstract', 'abstract_text', 'summary', 'summaries')
            if not paper and abstract:
                paper = f"{first(row, 'title', 'paper_title')}. {abstract}"
            if paper and abstract:
                pairs.append({'paper': paper, 'reference': abstract, 'source': json_path.name})
                if len(pairs) >= limit: return pairs
    return pairs

# Place the downloaded Kaggle dataset under DATASET_DIR for optional evaluation.
dataset_search_dirs = [DATASET_DIR]
dataset_pairs = []
for search_dir in dataset_search_dirs:
    dataset_pairs = load_kaggle_pairs(search_dir, limit=5)
    if dataset_pairs: break
if dataset_pairs:
    print(f'ArXiv dataset pairs discovered: {len(dataset_pairs)}')
else:
    print('ArXiv dataset not found; continuing in single-paper mode.')

## 1. PDF extraction and preprocessing

The extractor preserves page boundaries and removes common PDF artifacts. Scanned PDFs may require OCR before this notebook can process them.

In [ ]:
def extract_pdf_pages(pdf_path: Path) -> List[str]:
    # PyMuPDF preserves reading order and layout better for multi-column papers.
    document = pymupdf.open(str(pdf_path))
    return [page.get_text('text', sort=True) or '' for page in document]

def clean_text(text: str) -> str:
    text = text.replace('\u00ad', '')
    text = re.sub(r'(?<=\w)-\s*\n\s*(?=\w)', '', text)
    text = re.sub(r'\s*\n\s*', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_pdf_text(pdf_path: Path):
    pages = extract_pdf_pages(pdf_path)
    cleaned_pages = [clean_text(p) for p in pages]
    return '\n'.join(cleaned_pages), cleaned_pages

raw_text, page_text = extract_pdf_text(PDF_PATH)
print(f'Pages: {len(page_text)} | Characters: {len(raw_text):,} | Words: {len(raw_text.split()):,}')
print(raw_text[:900])

## 2. Section detection and long-document chunking

Standard encoder-decoder models cannot reliably consume an entire long paper at once. The implementation therefore detects common scientific sections, splits text into bounded chunks, summarizes chunks, and combines the results.

In [ ]:
SECTION_NAMES = [
    'abstract', 'introduction', 'background', 'related work', 'methodology',
    'methods', 'materials and methods', 'experiments', 'results',
    'discussion', 'conclusion', 'limitations', 'references'
]

def split_sentences(text: str) -> List[str]:
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9])', text)
    return [p.strip() for p in parts if len(p.strip()) > 20]

def detect_sections(text: str) -> Dict[str, str]:
    normalized = text
    matches = []
    for name in SECTION_NAMES:
        pattern = r'(?i)(?<![A-Za-z])' + re.escape(name) + r'(?![A-Za-z])'
        for m in re.finditer(pattern, normalized):
            prefix = normalized[max(0, m.start()-35):m.start()]
            if m.start() == 0 or prefix.endswith(('.', ':', ' ')):
                matches.append((m.start(), name))
    matches.sort()
    # Ignore headings accidentally matched inside the reference list.
    reference_index = next((i for i, (_, name) in enumerate(matches) if name == 'references'), None)
    if reference_index is not None:
        matches = matches[:reference_index + 1]
    sections = {}
    for i, (start, name) in enumerate(matches):
        end = matches[i+1][0] if i + 1 < len(matches) else len(normalized)
        content = normalized[start + len(name):end].strip(' :.-')
        if len(content.split()) >= 8 and name not in sections:
            sections[name] = content
    if not sections:
        sections['document'] = normalized
    return sections

def chunk_text(text: str, max_words: int = 650) -> List[str]:
    sentences = split_sentences(text)
    chunks, current, count = [], [], 0
    for sentence in sentences:
        words = len(sentence.split())
        if current and count + words > max_words:
            chunks.append(' '.join(current))
            current, count = [], 0
        current.append(sentence)
        count += words
    if current:
        chunks.append(' '.join(current))
    return chunks

sections = detect_sections(raw_text)
chunks = []
for section, content in sections.items():
    for chunk in chunk_text(content, MAX_CHUNK_WORDS):
        chunks.append({'section': section, 'text': chunk})
print('Detected sections:', list(sections))
print('Chunks:', len(chunks))

## 3. Summarization engine

The notebook uses a pretrained Transformer encoder-decoder model. Set `MODEL_NAME` to a compatible T5, BART, or PEGASUS checkpoint.

In [ ]:
class TransformerSummarizer:
    def __init__(self, model_name: str):
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        import torch
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model.to(self.device)
        self.model.eval()

    def summarize(self, text: str, max_new_tokens: int = 180) -> str:
        model_name = self.model.config.name_or_path.lower()
        prompt = text if ('bart' in model_name or 'pegasus' in model_name) else 'summarize: ' + text
        inputs = self.tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(self.device)
        output = self.model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4, no_repeat_ngram_size=3)
        return self.tokenizer.decode(output[0], skip_special_tokens=True).strip()

model = TransformerSummarizer(MODEL_NAME)
print('Loaded Transformer:', MODEL_NAME)

def summarize_chunks(chunk_records):
    summaries = []
    for record in chunk_records:
        summary = model.summarize(record['text'])
        summaries.append({'section': record['section'], 'summary': summary})
    combined = ' '.join(x['summary'] for x in summaries)
    final = model.summarize(combined)
    return final, summaries

document_summary, chunk_summaries = summarize_chunks(chunks)
print(document_summary)

## 4. Structured research-information extraction

The extractor returns the research objective, methodology, dataset used, key findings, and conclusion required by the assignment.

In [ ]:
FIELD_RULES = {
    'research_objective': ['objective', 'aim', 'we propose', 'we investigate', 'we study', 'this paper', 'goal'],
    'methodology': ['method', 'approach', 'architecture', 'algorithm', 'experiment', 'model', 'attention', 'encoder', 'decoder', 'trained'],
    'dataset_used': ['dataset', 'data set', 'corpus', 'benchmark', 'training data', 'sentence pairs', 'wmt', 'samples'],
    'key_findings': ['result', 'findings', 'achieve', 'outperform', 'improve', 'accuracy', 'significant', 'bleu', 'state-of-the-art'],
    'conclusion': ['conclusion', 'conclude', 'overall', 'future work', 'limitation', 'presented', 'proposed', 'faster']
}

FIELD_SECTIONS = {
    'research_objective': ['abstract', 'introduction'],
    'methodology': ['abstract', 'introduction', 'background', 'methods', 'methodology', 'experiments'],
    'dataset_used': ['methods', 'experiments', 'results'],
    'key_findings': ['results', 'experiments', 'conclusion'],
    'conclusion': ['conclusion', 'discussion']
}

def extract_structured_fields(text: str) -> Dict[str, Dict[str, str]]:
    # Search relevant scientific sections and exclude the reference list/metadata.
    all_sections = globals().get('sections', {})
    body = text.split(' References ', 1)[0]
    result = {}
    metadata = re.compile(r'^(provided proper attribution|.*arxiv:|.*nips 20\d\d|.*spent countless long days)', re.I)
    for field, keywords in FIELD_RULES.items():
        relevant = [all_sections[name] for name in FIELD_SECTIONS[field] if name in all_sections]
        search_text = ' '.join(relevant) if relevant else body
        sentences = [s for s in split_sentences(search_text) if not metadata.search(s)]
        scored = []
        for index, sentence in enumerate(sentences):
            lower = sentence.lower()
            hits = sum(1 for keyword in keywords if keyword in lower)
            if hits:
                scored.append((hits, -index, sentence))
        scored.sort(reverse=True)
        evidence = [item[2] for item in scored[:3]]
        if field == 'methodology':
            method_sentences = [s for s in sentences if not any(k in s.lower() for k in ['bleu', 'state-of-the-art', 'achieves', 'outperform'])]
            method_scored = [(sum(1 for k in keywords if k in s.lower()), -i, s) for i, s in enumerate(method_sentences)]
            evidence = [item[2] for item in sorted(method_scored, reverse=True)[:3] if item[0] > 0]
        if field == 'dataset_used':
            dataset_sentences = [s for s in split_sentences(body) if any(k in s.lower() for k in ['dataset', 'sentence pairs', 'training data', 'wmt 2014', 'corpus'])]
            dataset_scored = [(sum(1 for k in FIELD_RULES[field] if k in s.lower()), -i, s) for i, s in enumerate(dataset_sentences)]
            evidence = [item[2] for item in sorted(dataset_scored, reverse=True)[:3] if item[0] > 0]
        # Conclusions are often descriptive rather than keyword-heavy; use the section text.
        if field == 'conclusion' and not evidence and sentences:
            evidence = sentences[:3]
        # Dataset details may appear under a numbered training-data subsection.
        if field == 'dataset_used' and not evidence:
            dataset_sentences = [s for s in split_sentences(body) if any(k in s.lower() for k in FIELD_RULES[field])]
            evidence = dataset_sentences[:3]
        if not evidence:
            evidence = ['Information not confidently detected in the extracted text.']
        result[field] = {'value': ' '.join(evidence), 'evidence': evidence}
    return result

structured_fields = extract_structured_fields(raw_text)
for field, item in structured_fields.items():
    print('\n' + field + ':\n' + item['value'])

## 5. Evaluation metrics

The evaluation implements the five metrics listed in the assignment. Set `REFERENCE` to the author-provided summary or abstract.

In [ ]:
def words(text):
    return re.findall(r'\w+', text.lower())

def rouge_n(reference: str, candidate: str, n: int = 1) -> float:
    def grams(tokens):
        return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))
    ref, cand = grams(words(reference)), grams(words(candidate))
    overlap = sum((ref & cand).values())
    precision = overlap / max(1, sum(cand.values()))
    recall = overlap / max(1, sum(ref.values()))
    return 2 * precision * recall / max(1e-12, precision + recall)

def rouge_l(reference: str, candidate: str) -> float:
    a, b = words(reference), words(candidate)
    table = [[0] * (len(b)+1) for _ in range(len(a)+1)]
    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            table[i][j] = table[i-1][j-1] + 1 if a[i-1] == b[j-1] else max(table[i-1][j], table[i][j-1])
    lcs = table[-1][-1]
    precision, recall = lcs / max(1, len(b)), lcs / max(1, len(a))
    return 2 * precision * recall / max(1e-12, precision + recall)

def bleu(reference: str, candidate: str) -> float:
    # Standard 1-4 gram BLEU with clipped precision and brevity penalty.
    ref, cand = words(reference), words(candidate)
    if not cand: return 0.0
    precisions = []
    for n in range(1, 5):
        ref_grams = Counter(tuple(ref[i:i+n]) for i in range(len(ref)-n+1))
        cand_grams = Counter(tuple(cand[i:i+n]) for i in range(len(cand)-n+1))
        total = sum(cand_grams.values())
        clipped = sum((ref_grams & cand_grams).values())
        precisions.append(clipped / total if total else 0.0)
    if any(p == 0 for p in precisions): return 0.0
    geometric_mean = math.exp(sum(math.log(p) for p in precisions) / 4)
    brevity = 1.0 if len(cand) > len(ref) else math.exp(1 - len(ref) / max(1, len(cand)))
    return brevity * geometric_mean

def perplexity(source: str, reference: str) -> float:
    import torch
    source_inputs = model.tokenizer(source, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
    target_inputs = model.tokenizer(reference, return_tensors='pt', truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        loss = model.model(**source_inputs, labels=target_inputs['input_ids']).loss
    return float(torch.exp(loss).cpu())

def evaluate_summary(reference: str, candidate: str, source: str = None) -> Dict[str, float]:
    return {
        'rouge_1_f1': rouge_n(reference, candidate, 1),
        'rouge_2_f1': rouge_n(reference, candidate, 2),
        'rouge_l_f1': rouge_l(reference, candidate),
        'bleu_score': bleu(reference, candidate),
        'perplexity': perplexity(source if source is not None else raw_text, candidate)
    }

# The paper abstract is the author-provided reference summary.
REFERENCE = sections.get('abstract', '')
if not REFERENCE:
    raise ValueError('No abstract/reference summary was detected in the research paper.')
print(evaluate_summary(REFERENCE, document_summary))

## 6. Assignment compliance and dataset evaluation

The model is an encoder-decoder Transformer: the encoder represents the cleaned paper, self-attention captures contextual relationships, and the decoder generates the abstractive summary. The chunking stage makes the complete-paper workflow feasible within the model context window.

This notebook uses the Kaggle ArXiv Scientific Paper Dataset. Place the downloaded ArXiv CSV under `DATASET_DIR` to run the same pipeline over paper/abstract pairs. The single-paper run above remains a transparent demonstration.

In [ ]:
def evaluate_dataset(pairs, limit=5):
    """Evaluate ROUGE-1/2/L, BLEU, and perplexity on dataset paper/abstract pairs."""
    rows = []
    for index, pair in enumerate(pairs[:limit], start=1):
        pair_sections = detect_sections(pair['paper'])
        pair_chunks = [
            {'section': section, 'text': chunk}
            for section, content in pair_sections.items()
            for chunk in chunk_text(content, MAX_CHUNK_WORDS)
        ]
        prediction, _ = summarize_chunks(pair_chunks)
        metrics = evaluate_summary(pair['reference'], prediction, pair['paper'])
        rows.append({'paper': index, **metrics})
    return rows

if dataset_pairs:
    dataset_metrics = evaluate_dataset(dataset_pairs)
    print(*dataset_metrics, sep='\n')
else:
    print('Dataset evaluation skipped; single-paper summarization and metrics are available above.')